# 🏈 VFL Empire — Fresh Analysis Notebook
**Created:** 2026-06-12T07:56:36.644407Z  |  **Purpose:** Clean slate for new analysis (post Jun 11-12 aligned + odds DNA work)

## 🚀 How to Launch (recommended)
```bash
cd /home/ubuntu/faith-workspace/vfl-empire
source venv_betting/bin/activate
jupyter lab   # or: jupyter notebook
```

1. In Jupyter, select kernel **Python (vfl-empire venv)** (we just registered it).
2. Work with relative paths from the `vfl-empire` root.
3. First cell below will install any missing viz libs (matplotlib/seaborn).

**Key artifacts for this analysis:**
- `data/aligned/vfl_fixture_unified.csv` + `dataset_manifest.json` (154 seasons, ~61k rows)
- Postgres `vfl_empire.vfl_fixture_aligned` (same data + more live rows)
- `scripts/align_dataset.py` — run with `--refresh --export` after new data
- `research_logs/vfl_master_progress_log.md`, `vfl_100_percent_odds_dna.md`, `vfl_macro_odds_pandas.md`
- Recent pattern outputs: `data/micro_patterns.json`, `data/standings_patterns.json`, `unified_ml_matrix.parquet`
- `scripts/analyze_all_odds_categories.py` (the Jun 12 global binning script)

> We are starting **another analysis from scratch** while leveraging the solid aligned foundation and the 2-matchday-lag bulletproof oracle thesis.

## 1. Environment & Imports

In [ ]:
# Core
import pandas as pd
import numpy as np
from pathlib import Path
import json
import psycopg2
from contextlib import contextmanager
from datetime import datetime

# Viz (install if missing in this venv)
%pip install -q matplotlib seaborn plotly

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("✅ Environment ready")
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("Working dir:", Path.cwd())


## 2. Paths & Sanity Check

In [ ]:
BASE = Path.cwd()
assert (BASE / "data" / "aligned").exists(), "Run this notebook from the vfl-empire root directory"

ALIGNED_CSV = BASE / "data/aligned/vfl_fixture_unified.csv"
MANIFEST = BASE / "data/aligned/dataset_manifest.json"
MICRO_PATTERNS = BASE / "data/micro_patterns.json"
STANDINGS_PATTERNS = BASE / "data/standings_patterns.json"

print("Aligned CSV:", ALIGNED_CSV.exists(), ALIGNED_CSV.stat().st_size // 1024, "KB")
print("Manifest:", MANIFEST.exists())
print("Pattern files present:", MICRO_PATTERNS.exists(), STANDINGS_PATTERNS.exists())


## 3. Postgres Connection Helper (vfl_empire DB)

In [ ]:
DB_CONFIG = dict(
    dbname="vfl_empire",
    user="vfl_user",
    password="vfl_pass",
    host="localhost",
    port=5432,
)

@contextmanager
def get_conn():
    conn = psycopg2.connect(**DB_CONFIG)
    try:
        yield conn
    finally:
        conn.close()

def query_df(sql: str, params=None) -> pd.DataFrame:
    with get_conn() as conn:
        return pd.read_sql_query(sql, conn, params=params)

# Quick health check
try:
    df_test = query_df("SELECT COUNT(*) as n FROM vfl_fixture_aligned WHERE home_goals IS NOT NULL;")
    print("Postgres vfl_fixture_aligned (with results):", df_test.iloc[0, 0])
except Exception as e:
    print("PG connection issue (still OK if you prefer the CSV):", e)


## 4. Load the Aligned Dataset (preferred for speed: CSV)

In [ ]:
# Load the canonical aligned fixtures (one row per fixture with results + best normal odds)
df = pd.read_csv(ALIGNED_CSV)

# Light typing / derived columns (safe to extend)
if "season_name" not in df.columns and "season_id" in df.columns:
    # season_name may be stored; fall back if needed
    pass

print("Shape:", df.shape)
print("Columns:", list(df.columns)[:20], "..." if len(df.columns) > 20 else "")
print("\nFirst 3 rows:")
display(df.head(3))


## 5. Manifest & Season Overview

In [ ]:
with open(MANIFEST) as f:
    manifest = json.load(f)

print("Seasons in manifest:", manifest.get("seasons"))
print("Aligned at:", manifest.get("aligned_at"))
print("Join keys:", manifest.get("join_keys"))
print("Core odds fields:", manifest.get("core_odds_fields"))
print("\nRecent seasons (last 8):")
recent = manifest.get("season_names", [])[-8:]
for s in recent:
    print("  ", s)

# Quick season counts from the df (if season_name present)
if "season_name" in df.columns:
    season_counts = df.groupby("season_name").size().sort_index(ascending=False).head(10)
    print("\nTop recent season fixture counts (from CSV):")
    print(season_counts)
else:
    print("\nNote: 'season_name' column not in this CSV export — use season_id or load via PG for richer metadata.")


## 6. Basic Outcome Rates (baseline sanity)

In [ ]:
# Standard outcome flags if not already present
if "home_win" not in df.columns:
    df["home_win"] = (df.get("home_goals", 0) > df.get("away_goals", 0)).astype(int)
if "draw" not in df.columns:
    df["draw"] = (df.get("home_goals", 0) == df.get("away_goals", 0)).astype(int)
if "away_win" not in df.columns:
    df["away_win"] = (df.get("home_goals", 0) < df.get("away_goals", 0)).astype(int)

print("Overall (where results exist):")
mask = df.get("home_goals").notna() if "home_goals" in df else pd.Series(True, index=df.index)
print(df.loc[mask, ["home_win", "draw", "away_win"]].mean().round(4) * 100, "%")


## 7. Replicate / Extend Global Odds Category Analysis (GG + O2.5)

In [ ]:
# This mirrors the spirit of scripts/analyze_all_odds_categories.py + the Jun 12 research
# Filter to rows that have both results and the normal odds we care about
odds_cols = [c for c in ["gg", "o25", "o15", "u25", "u35", "ng"] if c in df.columns]
print("Available odds columns:", odds_cols)

work = df[df.get("home_goals").notna()].copy()
if "gg" in work.columns:
    work = work[work["gg"].notna()]

print("Matches with complete results + at least GG odds:", len(work))

# GG binning (0.05 steps like the production script)
if "gg" in work.columns:
    bins = np.arange(1.30, 2.55, 0.05)
    labels = [f"{b:.2f}-{b+0.04:.2f}" for b in bins[:-1]]
    work["gg_cat"] = pd.cut(work["gg"], bins=bins, labels=labels, right=False)
    
    gg_stats = (work.groupby("gg_cat", observed=True)
                .agg(matches=("gg", "count"),
                     home_win_pct=("home_win", "mean"),
                     draw_pct=("draw", "mean"),
                     o25_hit=("o25_hit", "mean") if "o25_hit" in work else ("home_goals", lambda x: np.nan))
                .reset_index())
    gg_stats["home_win_pct"] *= 100
    gg_stats = gg_stats[gg_stats["matches"] > 50].sort_values("gg_cat")
    print("\nGG Category → Home Win % (n>50):")
    print(gg_stats.to_string(index=False))


## 8. Load Recent Pattern Artifacts (micro + standings)

In [ ]:
if MICRO_PATTERNS.exists():
    with open(MICRO_PATTERNS) as f:
        micro = json.load(f)
    print("micro_patterns.json top-level keys:", list(micro.keys()) if isinstance(micro, dict) else type(micro))
    # Print a small sample
    if isinstance(micro, dict):
        for k in list(micro.keys())[:3]:
            print(f"  {k}:", str(micro[k])[:120] if not isinstance(micro[k], (list, dict)) else f"({len(micro[k])} items)")


## 9. Recent Seasons Focus (live window)

In [ ]:
# Example: focus on the most recent complete or near-complete seasons (VFLM 535x+)
if "season_name" in df.columns:
    recent_seasons = [s for s in df["season_name"].unique() if str(s).startswith("VFLM 535") or str(s).startswith("VFLM 536")]
    print("Recent season names found in data:", sorted(recent_seasons)[-5:])
    
    recent_df = df[df["season_name"].isin(recent_seasons)]
    print("Rows in recent seasons:", len(recent_df))
    
    # You can now run per-season or rolling analyses here
else:
    print("season_name not present — filter via season_id or re-export with richer columns from align_dataset.py")


## 10. Scratch / Next Analysis Area

Add your new work below. Ideas (from master progress + recent logs):

- Decision tree / sklearn pipeline on the full matrix for high-volume branches
- Proper train/holdout by season (avoid leakage)
- Incorporate standings / tier / phase features + the bulletproof locks
- Value betting scan with fresh edge threshold on the latest data
- PRNG / sequence features from recent captures
- Wire a pattern matcher that feeds the live oracle / rapid daemon
- Reproduce the 100% odds DNA clusters on the post-alignment data

Happy hunting — the 2-MD lag + odds DNA foundation is extremely strong.

In [ ]:
# === YOUR NEW ANALYSIS STARTS HERE ===

# Example starter: load the unified ML matrix if you want feature-rich work
# ml = pd.read_parquet(BASE / "data/unified_ml_matrix.parquet")
# print(ml.shape, ml.columns[:15])

# Or pull fresh from PG with more live rows:
# fresh = query_df("SELECT * FROM vfl_fixture_aligned ORDER BY season_id DESC, matchday_number DESC LIMIT 500;")

print("Notebook ready for new analysis from scratch. Delete or comment this cell when done.")
